# TabDPT Regressor — Artifact Inference Tutorial

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/tabdpt-regressor-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/tabdpt-regressor-pipeline/blob/main/tutorials/tabdpt_regressor_artifact_inference_colab.ipynb)
[![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Layer6%2FTabDPT-ffcc4d?style=flat)](https://huggingface.co/Layer6/TabDPT)
[![Upstream](https://img.shields.io/badge/Upstream-layer6ai--labs%2FTabDPT--inference-181717?style=flat&logo=github&logoColor=white)](https://github.com/layer6ai-labs/TabDPT-inference)
[![arXiv](https://img.shields.io/badge/arXiv-2608.01400-b31b1b.svg)](https://arxiv.org/abs/2608.01400)

This tutorial demonstrates how to load an exported DIMER serving artifact bundle (`artifact.json` + `training_context.csv`) in a fresh process, condition the in-context TabDPT foundation model on the saved support table, and run batch inference on new unlabelled regression records without refitting.

In [ ]:
!git clone -q https://github.com/kurtvalcorza/tabdpt-regressor-pipeline.git /content/tabdpt-regressor-pipeline
%pip install -q '/content/tabdpt-regressor-pipeline[model]'


## 1. Prepare and inspect the DIMER serving artifact bundle

In DIMER, a completed training/export run produces an `artifact.json` manifest referencing the support context table `training_context.csv`, the pinned base model metadata, and the versioned `preprocessing` state (`tabdpt-dimer-context-v2`).

In [ ]:
import hashlib
import json
from pathlib import Path
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from tabdpt_regressor_pipeline import (
    TABDPT_HF_REPO,
    TABDPT_HF_REVISION,
    TABDPT_UPSTREAM_CODE_COMMIT,
    TABDPT_WEIGHT_FILENAME,
    TABDPT_WEIGHT_SHA256,
    TabDPTRegressionPipeline,
    TabularFeatureEncoder,
)

# Prepare a sample dataset split for demonstration
frame = load_diabetes(as_frame=True).frame
train, test = train_test_split(frame, test_size=0.2, random_state=42)

# Train initial pipeline to export genuine DIMER artifact state
training_pipe = TabDPTRegressionPipeline(compile_model=False, use_flash=False)
training_pipe.fit(train, target_column='target')

bundle_dir = Path('/content/artifacts')
bundle_dir.mkdir(parents=True, exist_ok=True)
context_path = bundle_dir / 'training_context.csv'
train.to_csv(context_path, index=False)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b''):
            digest.update(chunk)
    return digest.hexdigest()

preprocessing_state = training_pipe.export_preprocessing_state()
manifest = {
    'format': 'tabdpt-dimer-context-v2',
    'taskType': 'tabular_regression',
    'targetColumn': 'target',
    'dropColumns': list(preprocessing_state['dropColumns']),
    'preprocessing': preprocessing_state,
    'baseModel': {
        'repo': TABDPT_HF_REPO,
        'revision': TABDPT_HF_REVISION,
        'filename': TABDPT_WEIGHT_FILENAME,
        'sha256': TABDPT_WEIGHT_SHA256,
        'upstreamCodeCommit': TABDPT_UPSTREAM_CODE_COMMIT,
    },
    'trainingContext': {
        'path': 'training_context.csv',
        'sha256': sha256_file(context_path),
    }
}
manifest_path = bundle_dir / 'artifact.json'
manifest_path.write_text(json.dumps(manifest, indent=2))
print('Serving artifact bundle initialized with v2 preprocessing state at:', bundle_dir)


## 2. Restore artifact bundle and condition the model without refitting

In DIMER production serving, the runtime must **never** re-fit preprocessing from `training_context.csv` (which would cause pandas `read_csv()` to mis-infer numeric-looking string categories like `"01"` as numbers).

Instead, the serving path restores the exact fitted feature schema and category maps from `manifest['preprocessing']` using `TabularFeatureEncoder.from_state()`, and conditions the foundation model on the saved support context without refitting. We demonstrate this via `TabDPTRegressionPipeline.load_artifact()` with explicit `use_flash=False`.

In [ ]:
# Load artifact bundle, verifying context digest and restoring preprocessing state via TabularFeatureEncoder.from_state()
pipe = TabDPTRegressionPipeline.load_artifact(
    manifest_path,
    compile_model=False,
    use_flash=False,
)
print('Serving pipeline restored from artifact without refitting!')
print('Target column:', pipe.target_column)
print('Restored feature count:', len(pipe.feature_encoder.feature_columns))


## 3. Score unlabelled observations

We drop the target column from the test set to simulate real-world unlabelled batch scoring, obtaining predicted continuous target values.

In [ ]:
unlabelled_test = test.drop(columns=[pipe.target_column])
predictions = pipe.predict(unlabelled_test, n_ensembles=2, context_size=512, batch_size=512, seed=42)

print('Predictions count:', len(predictions))
print('Sample predictions (first 5):')
print(predictions.head(5))


In [ ]:
# Evaluate against ground-truth continuous targets
metrics = pipe.evaluate(test, n_ensembles=2, context_size=512, batch_size=512, seed=42)
print('Holdout evaluation metrics:', metrics)


## Production notes

For production deployment in DIMER:
1. Because TabDPT is an in-context learner, the served model consists of `tabdpt1_2.safetensors` + `training_context.csv` + `artifact.json`.
2. Serving reloads restore fitted preprocessing state via `TabularFeatureEncoder.from_state()` rather than refitting from CSV, guaranteeing identical categorical codes and numeric assignments.
3. Treat `training_context.csv` with the same governance and security controls as the original training dataset.
4. Setting `use_flash=False` ensures portability across both Tesla T4 (`sm_75`) GPUs and newer Ampere/Hopper (`sm_80+`) architectures.
